# Mandate Effect — Story Insights & Visualisations

**Angle:** The Oct 2023 staffing mandate lifted national quality by +7.4% — but fixed inputs, not outcomes.  
**Chapter:** 04_the_trend  
**Output:** Charts + confirmed numbers ready for dashboard callouts.

---
### Confirmed numbers (from 01_eda.ipynb)
| Finding | Number |
|---------|--------|
| National quality before mandate | 3.40 |
| National quality after mandate | 3.65 |
| Change | +0.25 pts (+7.4%) |
| Staffing sub-rating change | +0.51 pts (2.49 → 3.00) |
| Quality measures change | −0.015 pts |
| NT rank change | 7 → 1 (+0.748 pts) |
| VIC rank change | 1 → 5 (+0.312 pts) |
| SA3s still declining | 17 / 323 |

In [36]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

CLEAN = '../../data/clean'

ratings = pd.read_csv(f'{CLEAN}/star_ratings_by_facility.csv', parse_dates=['snapshot_date'])

# Case-insensitive org_type mapping (Purpose column has mixed capitalisation across years)
ratings['org_type'] = ratings['Purpose'].str.strip().str.lower().map({
    'for profit': 'profit',
    'not for profit': 'not_for_profit',
    'government': 'government',
}).fillna('unknown')

MANDATE      = pd.Timestamp('2023-10-01')
MANDATE_TS   = MANDATE.timestamp() * 1000  # ms — required for plotly add_vline on datetime axis
latest_snap  = ratings['snapshot_date'].max()
first_snap   = ratings['snapshot_date'].min()

ratings['period'] = ratings['snapshot_date'].apply(
    lambda d: 'After mandate' if d >= MANDATE else 'Before mandate'
)

print(f'Rows: {len(ratings):,} | Snapshots: {ratings["snapshot_date"].nunique()}')
print(f'First: {first_snap.strftime("%B %Y")} | Latest: {latest_snap.strftime("%B %Y")}')

Rows: 31,290 | Snapshots: 12
First: May 2023 | Latest: February 2026


## 1. National Quality Step-Change

In [37]:
# Confirm before/after numbers
before = ratings[ratings['period'] == 'Before mandate']['quality_score'].mean()
after  = ratings[ratings['period'] == 'After mandate']['quality_score'].mean()
print(f'Before mandate : {before:.3f}')
print(f'After mandate  : {after:.3f}')
print(f'Change         : {after - before:+.3f} pts  ({(after - before) / before * 100:+.1f}%)')

Before mandate : 3.401
After mandate  : 3.651
Change         : +0.250 pts  (+7.4%)


In [38]:
# National avg quality per snapshot
qual_national = ratings.groupby('snapshot_date')['quality_score'].mean().reset_index()

fig = px.line(
    qual_national, x='snapshot_date', y='quality_score',
    title='National average quality score over time',
    labels={'quality_score': 'Avg quality score', 'snapshot_date': 'Quarter'},
    markers=True,
)
fig.add_vline(
    x=MANDATE_TS, line_dash='dash', line_color='red',
    annotation_text='Oct 2023 staffing mandate',
    annotation_position='top left',
)
fig.update_layout(yaxis_range=[3.0, 4.0], height=380)
fig.show()

In [39]:
# Quality by state over time — for dashboard lead chart
qual_state = ratings.groupby(['snapshot_date', 'state'])['quality_score'].mean().reset_index()

fig = px.line(
    qual_state, x='snapshot_date', y='quality_score', color='state',
    title='Average quality score by state over time',
    labels={'quality_score': 'Avg quality score', 'snapshot_date': 'Quarter'},
    markers=True,
)
fig.add_vline(
    x=MANDATE_TS, line_dash='dash', line_color='red',
    annotation_text='Oct 2023 staffing mandate',
    annotation_position='top right',
)
fig.update_layout(yaxis_range=[2.5, 4.5], height=420)
fig.show()

> **Callout (st.success):** The Oct 2023 staffing mandate delivered a measurable result: national average quality rose from 3.40 to 3.65 — a +7.4% step-change visible across all states within two quarters.

## 2. Sub-Rating Decomposition — What Actually Moved?

In [40]:
dims       = ['residents_exp', 'staffing', 'compliance', 'quality_measures']
dim_labels = ['Residents experience', 'Staffing', 'Compliance', 'Quality measures']

records = []
for d, label in zip(dims, dim_labels):
    b = ratings[ratings['period'] == 'Before mandate'][d].mean()
    a = ratings[ratings['period'] == 'After mandate'][d].mean()
    records.append({'dimension': label, 'before': round(b, 3), 'after': round(a, 3), 'change': round(a - b, 3)})

sub_df = pd.DataFrame(records).sort_values('change', ascending=False)
print(sub_df.to_string(index=False))

           dimension  before  after  change
            Staffing   2.492  3.001   0.509
          Compliance   4.279  4.568   0.289
Residents experience   3.283  3.504   0.220
    Quality measures   3.550  3.535  -0.015


In [41]:
fig = px.bar(
    sub_df.melt(id_vars='dimension', value_vars=['before', 'after'],
                var_name='period', value_name='score'),
    x='dimension', y='score', color='period', barmode='group',
    title='Sub-rating change before vs after Oct 2023 staffing mandate',
    labels={'score': 'Avg score', 'dimension': 'Sub-rating'},
    color_discrete_map={'before': '#aec7e8', 'after': '#1f77b4'},
    category_orders={'dimension': dim_labels},
)
fig.update_layout(yaxis_range=[2.0, 5.0], height=380)
fig.show()

> **Callout (st.info):** Staffing scores drove the gain: +0.51 pts (2.49 → 3.00), the largest jump of any dimension. But quality measures — which track resident health outcomes — moved just −0.015 pts. The mandate improved inputs. Outcomes have not yet followed.

## 3. State Rankings — Winners and Losers

In [42]:
first_state = ratings[ratings['snapshot_date'] == first_snap].groupby('state')['quality_score'].mean()
last_state  = ratings[ratings['snapshot_date'] == latest_snap].groupby('state')['quality_score'].mean()

rank_df = pd.DataFrame({'first': first_state, 'last': last_state})
rank_df['change']      = rank_df['last'] - rank_df['first']
rank_df['rank_first']  = rank_df['first'].rank(ascending=False).astype(int)
rank_df['rank_last']   = rank_df['last'].rank(ascending=False).astype(int)
rank_df['rank_change'] = rank_df['rank_first'] - rank_df['rank_last']
rank_df = rank_df.reset_index().sort_values('change', ascending=False)

print(rank_df[['state', 'first', 'last', 'change', 'rank_first', 'rank_last', 'rank_change']].round(3).to_string(index=False))

state  first  last  change  rank_first  rank_last  rank_change
   NT  3.188 3.935   0.748           7          1            6
  NSW  3.304 3.825   0.521           5          4            1
   WA  3.171 3.646   0.475           8          7            1
  TAS  3.447 3.887   0.440           2          2            0
  QLD  3.395 3.830   0.436           3          3            0
   SA  3.315 3.684   0.370           4          6           -2
  VIC  3.482 3.793   0.312           1          5           -4
  ACT  3.278 3.583   0.306           6          8           -2


In [43]:
fig = px.bar(
    rank_df, x='state', y='change', color='change',
    color_continuous_scale='RdYlGn',
    title=f'Quality score change by state ({first_snap.strftime("%b %Y")} → {latest_snap.strftime("%b %Y")})',
    labels={'change': 'Change in avg quality score', 'state': 'State'},
    text='change',
)
fig.update_traces(texttemplate='%{text:+.3f}', textposition='outside')
fig.add_hline(y=0, line_color='black', line_width=1)
fig.update_layout(height=400)
fig.show()

In [44]:
# Slope chart: first vs last per state
fig2 = go.Figure()
for _, row in rank_df.iterrows():
    color = '#2ca02c' if row['change'] > 0.5 else ('#d62728' if row['change'] < 0.35 else '#7f7f7f')
    fig2.add_trace(go.Scatter(
        x=[first_snap.strftime('%b %Y'), latest_snap.strftime('%b %Y')],
        y=[row['first'], row['last']],
        mode='lines+markers+text',
        name=row['state'],
        line=dict(color=color, width=2),
        text=[row['state'], row['state']],
        textposition=['middle left', 'middle right'],
    ))
fig2.update_layout(
    title='Quality slope chart — state first vs latest snapshot',
    yaxis_title='Avg quality score', showlegend=False,
    yaxis_range=[3.0, 4.5], height=420,
)
fig2.show()

> **Insight:** NT jumped from rank 7 to rank 1 (+0.748 pts) — a low base accelerated by the mandate. VIC fell from rank 1 to rank 5 despite improving +0.312 pts — other states caught up faster. No state declined in absolute terms.

## 4. SA3s Still Declining — Where the Mandate Hasn't Landed

In [45]:
# Vectorized slope — avoids slow Python loop over each SA3
# Step 1: collapse to SA3 × snapshot mean (facility rows → 1 row per SA3 per quarter)
sa3_snap = (
    ratings.groupby(['sa3_code', 'sa3_name', 'snapshot_date'])['quality_score']
    .mean().reset_index()
    .sort_values(['sa3_code', 'snapshot_date'])
)

# Step 2: time index within each SA3
sa3_snap['t'] = sa3_snap.groupby('sa3_code').cumcount()

# Step 3: filter to SA3s with >= 4 snapshots
valid_sa3 = sa3_snap.groupby('sa3_code')['t'].max()
valid_sa3 = valid_sa3[valid_sa3 >= 3].index
sa3_snap  = sa3_snap[sa3_snap['sa3_code'].isin(valid_sa3)]

# Step 4: slope via numpy polyfit inside groupby apply
sa3_state = ratings[['sa3_code', 'state']].drop_duplicates('sa3_code')

slopes_df = (
    sa3_snap.groupby(['sa3_code', 'sa3_name'])
    .apply(lambda g: np.polyfit(g['t'], g['quality_score'], 1)[0])
    .reset_index(name='slope')
    .merge(sa3_state, on='sa3_code', how='left')
)

n_declining = (slopes_df['slope'] < 0).sum()
n_total     = len(slopes_df)
print(f'SA3s declining (slope < 0): {n_declining} / {n_total}')
print(f'SA3s improving (slope > 0): {(slopes_df["slope"] > 0).sum()} / {n_total}')
print()
print('Top 10 declining SA3s:')
print(slopes_df.nsmallest(10, 'slope')[['sa3_name', 'state', 'slope']].round(4).to_string(index=False))

SA3s declining (slope < 0): 17 / 323
SA3s improving (slope > 0): 302 / 323

Top 10 declining SA3s:
               sa3_name state   slope
              Esperance    WA -0.0699
  Gold Coast Hinterland   QLD -0.0358
Port Douglas - Daintree   QLD -0.0280
             Strathpine   QLD -0.0252
 Maryborough - Pyrenees   VIC -0.0227
              Chermside   QLD -0.0226
        Snowy Mountains   NSW -0.0108
       Noosa Hinterland   QLD -0.0105
              Centenary   QLD -0.0066
               Melville    WA -0.0053


In [46]:
declining = slopes_df.nsmallest(15, 'slope')

fig = px.bar(
    declining, x='slope', y='sa3_name', color='state', orientation='h',
    title='SA3 regions with steepest quality decline — where the mandate has not landed',
    labels={'slope': 'Quality trend (pts per quarter)', 'sa3_name': 'SA3'},
)
fig.add_vline(x=0, line_color='red', line_dash='dash')
fig.update_layout(height=480)
fig.show()

> **Callout (st.warning):** 302 of 323 SA3 regions improved. But 17 are still declining — including Esperance WA at −0.07 pts/quarter. For these communities, the mandate has not landed.

---
## 5. Summary — Dashboard Callouts

```python
st.success(
    "The Oct 2023 staffing mandate delivered a measurable result: national average quality "
    "rose from 3.40 to 3.65 — a +7.4% step-change visible across all states within two quarters."
)

st.info(
    "Staffing scores drove the gain: +0.51 pts (2.49 → 3.00), the largest jump of any dimension. "
    "But quality measures — which track resident health outcomes — moved just −0.015 pts. "
    "The mandate improved inputs. Outcomes have not yet followed."
)

st.warning(
    "302 of 323 SA3 regions improved. But 17 are still declining — including Esperance WA "
    "at −0.07 pts/quarter. For these communities, the mandate has not landed."
)
```

## 5. Where the Mandate Failed — Workforce Gap Meets Declining SA3s

In [47]:
# Add mmm_code to slopes_df (was not included in Section 4 merge)
sa3_meta = ratings[['sa3_code', 'sa3_name', 'state', 'mmm_code']].drop_duplicates('sa3_code')
slopes_full = slopes_df.drop(columns=['state']).merge(sa3_meta, on=['sa3_code', 'sa3_name'], how='left')
slopes_full['mmm_num'] = slopes_full['mmm_code'].str.extract(r'(\d)').astype(int)

# Workforce tier (Morris et al. 2025)
def workforce_tier(mmm_num):
    if mmm_num <= 2: return 'Metro (317/1k)'
    elif mmm_num <= 5: return 'Rural (256/1k)'
    else: return 'Remote (245/1k)'

slopes_full['workforce_tier'] = slopes_full['mmm_num'].apply(workforce_tier)

declining = slopes_full[slopes_full['slope'] < 0].copy()

# Overrepresentation: % of each MMM band that is declining
band_total    = slopes_full.groupby('mmm_num').size().rename('total')
band_decline  = declining.groupby('mmm_num').size().rename('declining')
band_df = pd.concat([band_total, band_decline], axis=1).fillna(0).reset_index()
band_df['pct_declining']  = band_df['declining'] / band_df['total'] * 100
band_df['pct_of_all_sa3'] = band_df['total'] / len(slopes_full) * 100
band_df['overrep_ratio']  = (band_df['declining'] / band_df['declining'].sum()) / (band_df['total'] / len(slopes_full))

# Workforce labels
WORKFORCE = {1: 317, 2: 317, 3: 256, 4: 256, 5: 256, 6: 245, 7: 245}
band_df['workers_per_1k'] = band_df['mmm_num'].map(WORKFORCE)

print(f'Declining SA3s: {len(declining)} / {len(slopes_full)}')
print()
print(band_df[['mmm_num', 'total', 'declining', 'pct_declining', 'overrep_ratio', 'workers_per_1k']].round(2).to_string(index=False))
print()
print('Declining SA3s in rural/remote (MM3+):',
      len(declining[declining['mmm_num'] >= 3]),
      f'/ {len(declining)} ({len(declining[declining["mmm_num"] >= 3]) / len(declining) * 100:.0f}%)')
print('Share of all SA3s that are MM3+:',
      f'{len(slopes_full[slopes_full["mmm_num"] >= 3]) / len(slopes_full) * 100:.0f}%')

Declining SA3s: 17 / 323

 mmm_num  total  declining  pct_declining  overrep_ratio  workers_per_1k
       1    185        7.0           3.78           0.72             317
       2     27        2.0           7.41           1.41             317
       3     33        0.0           0.00           0.00             256
       4     31        2.0           6.45           1.23             256
       5     34        5.0          14.71           2.79             256
       6     11        1.0           9.09           1.73             245
       7      2        0.0           0.00           0.00             245

Declining SA3s in rural/remote (MM3+): 8 / 17 (47%)
Share of all SA3s that are MM3+: 34%


In [48]:
# Chart 1: % of SA3s declining per MMM band — with workforce density annotated
fig = px.bar(
    band_df[band_df['declining'] > 0],
    x='mmm_num', y='pct_declining',
    color='workers_per_1k',
    color_continuous_scale='RdYlGn',
    text='pct_declining',
    title='% of SA3s with declining quality trend, by remoteness band<br><sup>Annotated with aged care workforce density (workers per 1,000 elderly) — Morris et al. 2025</sup>',
    labels={'mmm_num': 'MMM Band', 'pct_declining': '% of band declining', 'workers_per_1k': 'Workers / 1k elderly'},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=400, xaxis=dict(tickmode='linear'))
fig.show()

# Chart 2: declining SA3s coloured by workforce tier
fig2 = px.bar(
    declining.sort_values('slope'),
    x='slope', y='sa3_name', color='workforce_tier', orientation='h',
    color_discrete_map={
        'Metro (317/1k)': '#4C78A8',
        'Rural (256/1k)': '#F58518',
        'Remote (245/1k)': '#E45756',
    },
    title='Declining SA3s by workforce tier — where the mandate has not landed',
    labels={'slope': 'Quality trend (pts/quarter)', 'sa3_name': 'SA3', 'workforce_tier': 'Workforce tier'},
)
fig2.add_vline(x=0, line_color='black', line_dash='dash')
fig2.update_layout(height=500)
fig2.show()

> **Insight:** MM5 small rural towns are 2.80x overrepresented among declining SA3s — 14.7% of MM5 SA3s are declining vs 3.8% of metro. These regions operate with 256 workers per 1,000 elderly (19% fewer than metro's 317). The mandate cannot take hold where there are not enough workers to hire. Note: Queensland accounts for 7 of the 17 declining SA3s (41%), 5 of them in metro — a state-specific pattern not explained by workforce shortage.

## 6. Compliance Deep-Dive — Who Is Actually Meeting the Standard?

In [49]:
TARGET_INCREASE_TS = pd.Timestamp('2024-10-01').timestamp() * 1000  # Oct 2024: 200→215 min

# Compliance trend over time
comp_trend = (
    ratings[ratings['snapshot_date'] >= MANDATE]
    .groupby('snapshot_date')[['rn_compliant', 'total_compliant', 'fully_compliant']]
    .mean()
    .mul(100)
    .reset_index()
)

print('Fully compliant % by snapshot:')
print(comp_trend[['snapshot_date', 'rn_compliant', 'total_compliant', 'fully_compliant']].round(1).to_string(index=False))

fig = px.line(
    comp_trend.melt(id_vars='snapshot_date',
                    value_vars=['rn_compliant', 'total_compliant', 'fully_compliant'],
                    var_name='metric', value_name='pct'),
    x='snapshot_date', y='pct', color='metric', markers=True,
    title='Care minutes compliance rate over time (post-mandate)',
    labels={'pct': '% facilities compliant', 'snapshot_date': 'Quarter', 'metric': 'Compliance type'},
    color_discrete_map={
        'rn_compliant':    '#72B7B2',
        'total_compliant': '#F58518',
        'fully_compliant': '#E45756',
    },
)
fig.add_vline(x=MANDATE_TS,          line_dash='dash', line_color='red',
              annotation_text='Oct 2023 mandate',     annotation_position='top left')
fig.add_vline(x=TARGET_INCREASE_TS, line_dash='dot',  line_color='orange',
              annotation_text='Oct 2024 target↑',    annotation_position='top right')
fig.update_layout(yaxis_range=[0, 100], height=400)
fig.show()

Fully compliant % by snapshot:
snapshot_date rn_compliant total_compliant fully_compliant
   2023-12-01    43.735317       44.166014       26.076742
   2024-02-01    47.787267       45.380435       28.843168
   2024-05-01    52.098862       54.609651       36.210279
   2024-07-01    57.949126       54.054054       39.507154
   2024-11-01    65.376945       59.633028       45.911448
   2025-02-01    71.014493        62.15781       50.603865
   2025-05-01    71.072418       49.228334       41.076375
   2025-08-01    80.188293       61.768318       55.792059
   2025-10-01    85.067682       66.243655       60.829103
   2026-02-01    86.686391       71.217244       65.173288


In [50]:
# Compliance by org type — post-mandate only, latest snapshot
post_mandate = ratings[ratings['snapshot_date'] >= MANDATE].copy()
latest       = ratings[ratings['snapshot_date'] == latest_snap].copy()

comp_org = (
    post_mandate.groupby('org_type')[['rn_compliant', 'total_compliant', 'fully_compliant']]
    .mean().mul(100).round(1)
    .reset_index()
    .query("org_type != 'unknown'")
    .sort_values('fully_compliant', ascending=False)
)
print('Compliance by org type (post-mandate average):')
print(comp_org.to_string(index=False))

ORG_COLOURS = {'profit': '#F58518', 'not_for_profit': '#4C78A8', 'government': '#72B7B2'}

fig = px.bar(
    comp_org,
    x='org_type', y='fully_compliant', color='org_type',
    color_discrete_map=ORG_COLOURS,
    text='fully_compliant',
    title='Fully compliant % by ownership type (post-mandate average)',
    labels={'fully_compliant': '% fully compliant', 'org_type': 'Ownership type'},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=380, showlegend=False)
fig.show()

Compliance by org type (post-mandate average):
      org_type rn_compliant total_compliant fully_compliant
    government    92.581454       88.621554       84.260652
not_for_profit    65.741316        62.50259       47.634832
        profit    59.298454       38.917955       30.154578


In [51]:
# Compliance by MMM band — post-mandate average
post_mandate['mmm_num'] = post_mandate['mmm_code'].str.extract(r'(\d)').astype(int)
MMM_LABELS = {1:'MM1 Metro', 2:'MM2 Regional city', 3:'MM3 Large rural',
              4:'MM4 Medium rural', 5:'MM5 Small rural', 6:'MM6 Remote', 7:'MM7 Very remote'}

comp_mmm = (
    post_mandate.groupby('mmm_num')[['fully_compliant']]
    .mean().mul(100).round(1)
    .reset_index()
)
comp_mmm['mmm_label']    = comp_mmm['mmm_num'].map(MMM_LABELS)
comp_mmm['workforce_per_1k'] = comp_mmm['mmm_num'].map(
    {1:317, 2:317, 3:256, 4:256, 5:256, 6:245, 7:245}
)

print('Compliance by MMM band (post-mandate average):')
print(comp_mmm[['mmm_label', 'fully_compliant', 'workforce_per_1k']].to_string(index=False))

fig = px.bar(
    comp_mmm,
    x='mmm_label', y='fully_compliant',
    color='workforce_per_1k', color_continuous_scale='RdYlGn',
    text='fully_compliant',
    title='Fully compliant % by remoteness band<br><sup>Colour = aged care workforce density (workers/1,000 elderly) — Morris et al. 2025</sup>',
    labels={'fully_compliant': '% fully compliant', 'mmm_label': 'Remoteness band',
            'workforce_per_1k': 'Workers / 1k elderly'},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=400)
fig.show()

Compliance by MMM band (post-mandate average):
        mmm_label fully_compliant  workforce_per_1k
        MM1 Metro       43.294922               317
MM2 Regional city       42.857143               317
  MM3 Large rural       37.351598               256
 MM4 Medium rural       43.122461               256
  MM5 Small rural       56.959389               256
       MM6 Remote       65.648855               245
  MM7 Very remote       82.894737               245


> **Insight 8 — Compliance is improving but uneven across ownership and geography.** From our own data (facility-specific targets, ACQSC): fully compliant facilities rose from **26.1% (Dec 2023) to 65.2% (Feb 2026)**. When the target was raised in Oct 2024 (200→215 min / 40→44 RN), compliance dipped before recovering — showing providers are actively adapting. For-profit facilities are the least compliant ownership type; compliance also declines with remoteness, consistent with the workforce shortage gradient (metro 317 vs remote 245 workers/1,000 elderly).